Chatbot
### AI that remembers your conversation!
---
**Model:** facebook/blenderbot-400M-distill



---
## Step 1 — Install and load the chatbot


In [1]:
!pip install transformers gradio -q
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity_error()

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

print('Loading chatbot model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForCausalLM.from_pretrained(MODEL)
print('Chatbot ready!')

Loading chatbot model...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Chatbot ready!


---
## Step 2 — Create the chat function

> We store the conversation as a simple list.
> Every message (yours + bot) gets added to that list.
> The bot reads the whole list each time — that is its memory!


In [2]:
def chat(user_message, history):
    '''
    user_message : what you typed
    history      : list of all messages so far
    '''
    # Add your message to history
    history.append({'role': 'user', 'content': user_message})

    # Build the full conversation text for the model
    # Each message separated by a special token
    conversation = tokenizer.sep_token.join(
        [m['content'] for m in history]
    )

    # Tokenize and generate reply
    inputs = tokenizer(
        conversation,
        return_tensors = 'pt',
        truncation     = True,
        max_length     = 128
    )
    output = model.generate(
        **inputs,
        max_new_tokens = 60
    )
    bot_reply = tokenizer.decode(output[0], skip_special_tokens=True)

    # Add bot reply to history
    history.append({'role': 'bot', 'content': bot_reply})

    return bot_reply, history

print('chat() function ready!')


chat() function ready!


---
## Step 3 — Send one message


In [3]:
# Redefining chat function to support Qwen's chat template
def chat(user_message, history):
    # Add user message
    history.append({'role': 'user', 'content': user_message})

    # Use Qwen's built-in chat template instead of manual joining
    conversation = tokenizer.apply_chat_template(
        history,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(conversation, return_tensors='pt')
    output = model.generate(**inputs, max_new_tokens=128)

    # Only decode the newly generated tokens
    generated_tokens = output[0][inputs.input_ids.shape[-1]:]
    bot_reply = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    # Add bot reply to history
    history.append({'role': 'assistant', 'content': bot_reply})
    return bot_reply, history

# Start with empty history
history = []

# Send first message
reply, history = chat('What is artificial intelligence?', history)

print('You :', 'What is artificial intelligence?')
print('Bot :', reply)

You : What is artificial intelligence?
Bot : Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are designed to think and learn like humans. It involves developing systems that can perform tasks that typically require human intelligence, such as perception, learning, reasoning, problem-solving, and decision-making.

Here's a breakdown of what AI entails:

1. **Machine Learning**: A subset of AI that allows computers to improve their performance on a task through experience without being explicitly programmed.
2. **Natural Language Processing (NLP)**: The ability of AI systems to understand and interpret natural language. This includes tasks like sentiment analysis, language translation, and generating human-like text


---
## Step 4 — See the memory in action!

> Tell the bot something personal.
> Then ask a follow-up — it should remember!


In [4]:
# Fresh start
history = []

# Message 1 — tell it something
reply1, history = chat('My favourite sport is cricket.', history)
print('You : My favourite sport is cricket.')
print('Bot :', reply1)
print()

# Message 2 — ask a follow-up
reply2, history = chat('What sport did I just mention?', history)
print('You : What sport did I just mention?')
print('Bot :', reply2)
print()
print('The bot should remember cricket!')


You : My favourite sport is cricket.
Bot : As an artificial intelligence language model, I don't have personal preferences or emotions like humans do. However, I can tell you that cricket is one of the most popular sports worldwide, and it's played by millions of people around the world. It involves running, batting, bowling, fielding, and various strategies to score runs and win the game. Cricket has a rich history dating back thousands of years, and it continues to be an important part of many cultures around the world. If you're interested in learning more about cricket, there are many resources available online for beginners and experienced players alike.

You : What sport did I just mention?
Bot : You mentioned "cricket." As an AI language model, I am not capable of directly mentioning specific sporting events or games, but I can provide information on different types of cricket if that would be helpful.

The bot should remember cricket!


---
## Step 5 — Have a full conversation


In [5]:
# Fresh start
history = []

questions = [
    'Tell me about smart cities.',
    'What sensors are used in smart cities?',
    'How does AI help manage traffic?',
    'What are the benefits for citizens?',
]

print('Full conversation about Smart Cities:')
print('=' * 45)
print()

for i, question in enumerate(questions, 1):
    reply, history = chat(question, history)
    print(f'Turn {i}')
    print(f'You : {question}')
    print(f'Bot : {reply}')
    print()


Full conversation about Smart Cities:

Turn 1
You : Tell me about smart cities.
Bot : Smart cities refer to urban areas that have been designed and operated with the aid of advanced technologies such as artificial intelligence, big data, and Internet of Things (IoT) technology. These technologies enable cities to better manage resources, improve services, and enhance quality of life for residents.

Here are some key features and benefits of smart cities:

1. Efficient Resource Management: Smart cities use IoT devices to monitor energy consumption, waste management, and traffic flow in real-time. This helps in optimizing resource allocation and reducing waste, which can save money and reduce carbon emissions.

2. Enhanced Services: With AI-powered chatbots and virtual assistants, smart cities

Turn 2
You : What sensors are used in smart cities?
Bot : In smart cities, various sensors are deployed across different sectors to gather data on urban environments and behaviors. Some common typ

---
## Step 6 — Try your own topic!

> ✏️ Change the questions to anything you like!


In [6]:
# ✏️ Change these questions!
history = []

my_questions = [
    'Tell me about machine learning.',
    'How is it different from regular programming?',
    'Can you give me a simple example?',
]

for i, question in enumerate(my_questions, 1):
    reply, history = chat(question, history)
    print(f'You : {question}')
    print(f'Bot : {reply}')
    print()


You : Tell me about machine learning.
Bot : Machine Learning is a subset of Artificial Intelligence that involves the development of algorithms and statistical models that enable computers to learn from data without being explicitly programmed. It focuses on enabling systems to improve their performance based on experience or feedback, rather than being pre-programmed.

Key components of Machine Learning include:

1. **Data**: The raw information used to train the model.
2. **Models**: These are mathematical representations of relationships between input variables (features) and output variable(s) (targets). There are two main types: supervised learning where the target variable is known, and unsupervised learning where it's unknown.
3. **Techniques**

You : How is it different from regular programming?
Bot : Machine Learning differs significantly from traditional programming in several key ways:

1. **Data-Driven Nature**: Traditional programming languages like Python use a syntax des

---
## Step 7 — Build a Gradio Chatbot App

> Type any message — bot replies and remembers!
> Click **Clear** to start a fresh conversation.


In [ ]:

import gradio as gr
from datetime import datetime

# Conversation storage
saved_chats = {}
active_chat = {"id": "chat_1"}
chat_counter = {"value": 1}

def get_active_history():
    chat_id = active_chat["id"]
    if chat_id not in saved_chats:
        saved_chats[chat_id] = {
            "title": "New Chat",
            "messages": [],
            "model_history": []
        }
    return saved_chats[chat_id]


def respond(message, displayed_history):
    message = (message or "").strip()
    if not message:
        return displayed_history or [], ""

    current = get_active_history()

    try:
        # Call your existing Qwen chat function
        reply, updated_history = chat(
            message,
            current["model_history"]
        )

        current["model_history"] = updated_history
        reply = str(reply).strip()

        if current["title"] == "New Chat":
            current["title"] = message[:28]

        current["messages"].append(
            {"role": "user", "content": message}
        )
        current["messages"].append(
            {"role": "assistant", "content": reply}
        )

        return current["messages"].copy(), ""

    except Exception as e:
        print("DBot error:", repr(e))
        return (
            (displayed_history or []) + [
                {
                    "role": "assistant",
                    "content": "Sorry! Something went wrong. Please try again."
                }
            ],
            ""
        )


def new_chat():
    chat_counter["value"] += 1
    chat_id = "chat_" + str(chat_counter["value"])

    active_chat["id"] = chat_id
    saved_chats[chat_id] = {
        "title": "New Chat",
        "messages": [],
        "model_history": []
    }

    return [], "", gr.update(
        choices=get_chat_choices(),
        value=chat_id
    )


def clear_current_chat():
    current = get_active_history()
    current["messages"] = []
    current["model_history"] = []
    current["title"] = "New Chat"
    return [], ""


def get_chat_choices():
    return [
        (item["title"], chat_id)
        for chat_id, item in saved_chats.items()
    ]


def load_chat(chat_id):
    if not chat_id or chat_id not in saved_chats:
        return [], ""

    active_chat["id"] = chat_id
    return saved_chats[chat_id]["messages"].copy(), ""


def use_prompt(prompt):
    return prompt


# Gradient theme CSS


css = """
/* Clean white and blue theme */
.gradio-container {
    background: linear-gradient(
        135deg, #F8FAFF 0%, #EEF4FF 55%, #E4EEFF 100%
    ) !important;
    color: #1E293B !important;
}

/* Header */
#dbot-title {
    text-align: center !important;
    font-size: 40px !important;
    font-weight: 800 !important;
    color: #2563EB !important;
    -webkit-text-fill-color: #2563EB !important;
    margin-bottom: 4px !important;
}

#dbot-subtitle {
    text-align: center !important;
    color: #64748B !important;
    font-size: 15px !important;
}

/* Text and headings */
.gradio-container h1,
.gradio-container h2,
.gradio-container h3,
.gradio-container p,
.gradio-container label,
.gradio-container .markdown {
    color: #1E293B !important;
}

/* Chat window */
#chat-area,
#chat-area > div,
#chat-area .wrap,
#chat-area .message-wrap,
#chat-area .scroll-hide,
#chat-area .wrapper {
    background: #FFFFFF !important;
    color: #1E293B !important;
    border-color: #D9E5F5 !important;
}

#chat-area {
    border: 1px solid #D9E5F5 !important;
    border-radius: 18px !important;
    overflow: hidden !important;
}

/* Chat messages */
#chat-area .message,
#chat-area .bubble,
#chat-area .message-bubble-border {
    color: #1E293B !important;
    border-radius: 14px !important;
}

/* Input */
#message-box,
#message-box > div,
#message-box .wrap,
#message-box label {
    background: transparent !important;
    color: #334155 !important;
}

#message-box textarea,
#message-box input {
    background: #FFFFFF !important;
    color: #1E293B !important;
    border: 1px solid #CBD9EE !important;
    border-radius: 14px !important;
    padding: 14px !important;
    box-shadow: 0 2px 8px rgba(37, 99, 235, 0.05) !important;
}

#message-box textarea::placeholder {
    color: #94A3B8 !important;
}

/* Buttons */
#send-button,
#new-chat-button {
    background: linear-gradient(
        100deg, #2563EB, #3B82F6
    ) !important;
    color: #FFFFFF !important;
    border: none !important;
    border-radius: 12px !important;
    font-weight: 700 !important;
}

#send-button:hover,
#new-chat-button:hover {
    background: #1D4ED8 !important;
}

/* Suggested prompts */
#prompt-row button {
    background: #FFFFFF !important;
    color: #1D4ED8 !important;
    border: 1px solid #C7D9F8 !important;
    border-radius: 12px !important;
    min-height: 44px !important;
}

#prompt-row button:hover {
    background: #EFF6FF !important;
    border-color: #60A5FA !important;
}

/* Chat history and other controls */
.gradio-container input,
.gradio-container textarea {
    color: #1E293B !important;
}

.gradio-container [role="radio"],
.gradio-container [role="listbox"],
.gradio-container .wrap-inner {
    background-color: #FFFFFF !important;
    color: #1E293B !important;
}

/* Footer */
footer {
    display: none !important;
}

/* Mobile */
@media (max-width: 768px) {
    #dbot-title {
        font-size: 32px !important;
    }

    #chat-area {
        min-height: 350px !important;
    }
}
"""



with gr.Blocks(
    css=css,
    title="DBot - AI Assistant",
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="sky",
        neutral_hue="slate"
    )
) as app:

    gr.Markdown("# ✨ D-Bot", elem_id="dbot-title")
    gr.Markdown(
        "Your AI companion, powered by Qwen",
        elem_id="dbot-subtitle"
    )

    with gr.Row():

        # Sidebar: saved conversations
        with gr.Column(scale=1, min_width=180):
            gr.Markdown("### 💬 Conversations")

            new_btn = gr.Button(
                "＋ New Chat",
                elem_id="new-chat-button"
            )

            history_picker = gr.Radio(
                choices=get_chat_choices(),
                label="Chat history",
                value="chat_1",
                interactive=True
            )

            clear_btn = gr.Button("🗑️ Clear Current Chat")

            gr.Markdown(
                "### About D-Bot 🤖\n"
                "Ask questions, learn concepts, "
                "or explore new ideas."
            )

        # Main chat
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(
                label="DBot Conversation",
                height=440,
                elem_id="chat-area"
            )

            gr.Markdown("### ✨ Try asking")

            with gr.Row(elem_id="prompt-row"):
                prompt1 = gr.Button("Explain GenAI")
                prompt2 = gr.Button("Help with Python")
                prompt3 = gr.Button("Tell me a fun fact")

            with gr.Row():
                msg = gr.Textbox(
                    placeholder="Message DBot...",
                    show_label=False,
                    lines=2,
                    elem_id="message-box",
                    scale=5
                )
                send = gr.Button(
                    "➤ Send",
                    variant="primary",
                    elem_id="send-button",
                    scale=1
                )

            gr.Markdown(
                "<center>DBot may make mistakes. "
                "Check important information.</center>"
            )

    # Send a message
    send.click(
        respond,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )

    msg.submit(
        respond,
        inputs=[msg, chatbot],
        outputs=[chatbot, msg]
    )

    # Suggested prompts
    prompt1.click(
        lambda: "Explain Generative AI in simple English with an example.",
        outputs=msg
    )
    prompt2.click(
        lambda: "Help me understand Python with a simple example.",
        outputs=msg
    )
    prompt3.click(
        lambda: "Tell me an interesting fun fact.",
        outputs=msg
    )

    # Start a fresh conversation
    new_btn.click(
        new_chat,
        outputs=[chatbot, msg, history_picker]
    )

    # Clear current conversation
    clear_btn.click(
        clear_current_chat,
        outputs=[chatbot, msg]
    )

    # Switch between saved conversations
    history_picker.change(
        load_chat,
        inputs=history_picker,
        outputs=[chatbot, msg]
    )

app.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5500d672f43c4e2580.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
